
# RQ2 — Fed-CGM Federated Configuration Tuning


In [1]:
from pathlib import Path
import sys
import json
import random
import numpy as np
import pandas as pd
import torch

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/dissertation")
else:
    PROJECT_ROOT = Path("../../..").resolve()

assert PROJECT_ROOT.exists(), f"PROJECT_ROOT does not exist: {PROJECT_ROOT}"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import dataloader as data
from src import federated as fed
from src.models.nn_baseline import GRUPredictor, HORIZONS

print("IN_COLAB:", IN_COLAB)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


Mounted at /content/drive
IN_COLAB: True
PROJECT_ROOT: /content/drive/MyDrive/dissertation
PyTorch: 2.11.0+cu128
CUDA available: True


## 1. Tuning design

In [2]:
TUNING_SEEDS = [7, 14, 21, 28, 35, 42, 49, 56, 63, 70]

LOCAL_EPOCH_GRID = [1, 2, 5]
CLIENTS_PER_ROUND_GRID = [15, 30]
EARLY_STOP_PATIENCE_GRID = [10, 20]

# Locked RQ1 model settings
GRU_HIDDEN_SIZE = 256
GRU_NUM_LAYERS = 3
GRU_DROPOUT = 0.2

# Locked optimisation settings
LR = 1e-4
WEIGHT_DECAY = 1e-5
BATCH_SIZE = 256
MAX_GRAD_NORM = 1.0

# Fixed FL training controls
MAX_ROUNDS = 100
LR_PATIENCE = 5
LR_FACTOR = 0.5

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# CRITICAL: absolute Drive-backed output path.
TUNING_DIR = (
    PROJECT_ROOT
    / "experiments"
    / "7_federated_learning"
    / "results"
    / "configuration_tuning"
    / "fed_cgm_multiseed"
)
TUNING_DIR.mkdir(parents=True, exist_ok=True)

RUN_RESULTS_PATH = TUNING_DIR / "run_level_results.csv"
HISTORIES_PATH = TUNING_DIR / "all_histories.json"
SUMMARY_PATH = TUNING_DIR / "configuration_summary_across_seeds.csv"
SELECTED_PATH = TUNING_DIR / "selected_fl_configuration.json"

N_CONFIGS = (
    len(LOCAL_EPOCH_GRID)
    * len(CLIENTS_PER_ROUND_GRID)
    * len(EARLY_STOP_PATIENCE_GRID)
)
N_RUNS = N_CONFIGS * len(TUNING_SEEDS)

print("Device:", DEVICE)
print("Configurations:", N_CONFIGS)
print("Seeds/configuration:", len(TUNING_SEEDS))
print("Total runs:", N_RUNS)
print("Drive output directory:", TUNING_DIR)
print("Drive-backed:", str(TUNING_DIR).startswith(str(PROJECT_ROOT)))

assert str(TUNING_DIR).startswith(str(PROJECT_ROOT)), (
    "TUNING_DIR is not inside PROJECT_ROOT. Refusing to run."
)


Device: cuda
Configurations: 12
Seeds/configuration: 10
Total runs: 120
Drive output directory: /content/drive/MyDrive/dissertation/experiments/7_federated_learning/results/configuration_tuning/fed_cgm_multiseed
Drive-backed: True


## 2. Reproducibility

In [3]:

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    if hasattr(torch.backends, "cudnn"):
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


## 3. Load aligned train clients and validation data

In [5]:

stats = data.load_cgm_stats()

client_loaders = fed.load_federated_clients(
    "train",
    encoder_name=None,
    batch_size=BATCH_SIZE,
)

val_loader = data.load_multimodal_tensors(
    "val",
    encoder_name=None,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

assert len(client_loaders) == 30, (
    f"Expected 30 training clients, got {len(client_loaders)}."
)

print("Training clients:", len(client_loaders))
print("Federated train samples:",
      sum(len(loader.dataset) for loader in client_loaders.values()))
print("Validation samples:", len(val_loader.dataset))
print("CGM mean:", stats["cgm_mean"])
print("CGM std:", stats["cgm_std"])


Training clients: 30
Federated train samples: 82681
Validation samples: 16355
CGM mean: 140.10437936402963
CGM std: 42.125711031429994


## 4. Verify common aligned sample population

In [6]:

aligned_train_samples = data.load_multimodal_samples("train")

n_aligned = len(aligned_train_samples)
n_federated = sum(len(loader.dataset) for loader in client_loaders.values())

assert not any(
    s["has_meal"] and not s["has_image"]
    for s in aligned_train_samples
), "Aligned population still contains meal-associated samples without images."

assert n_aligned == n_federated, (
    f"Sample mismatch: aligned={n_aligned}, federated={n_federated}"
)

expected_by_pid = {}
for sample in aligned_train_samples:
    pid = sample["participant_id"]
    expected_by_pid[pid] = expected_by_pid.get(pid, 0) + 1

actual_by_pid = {
    pid: len(loader.dataset)
    for pid, loader in client_loaders.items()
}

assert expected_by_pid == actual_by_pid, (
    "Participant-level counts differ between aligned samples and FL clients."
)

print("Sample alignment: PASS")
print("Aligned train samples:", n_aligned)
print("Federated train samples:", n_federated)


Sample alignment: PASS
Aligned train samples: 82681
Federated train samples: 82681


## 5. Locked Fed-CGM model

In [7]:

def model_fn():
    return GRUPredictor(
        hidden_size=GRU_HIDDEN_SIZE,
        num_layers=GRU_NUM_LAYERS,
        dropout=GRU_DROPOUT,
        num_horizons=len(HORIZONS),
    )


def forward_fn(model, batch, device):
    cgm, tsm, has_meal, target = batch
    return model(cgm.to(device)), target.to(device)


test_model = model_fn()
print(test_model)
print("Trainable parameters:",
      sum(p.numel() for p in test_model.parameters() if p.requires_grad))


GRUPredictor(
  (gru): GRU(1, 256, num_layers=3, batch_first=True, dropout=0.2)
  (head): Sequential(
    (0): Linear(in_features=256, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=128, out_features=5, bias=True)
  )
)
Trainable parameters: 1021957


## 6. Result helpers

In [8]:

def summarise_run(
    history,
    seed,
    local_epochs,
    clients_per_round,
    early_stop_patience,
    config_name,
    checkpoint_dir,
):
    val_loss = np.asarray(history["val_loss"], dtype=float)
    val_rmse = np.asarray(history["val_rmse"], dtype=float)
    val_mae = np.asarray(history["val_mae"], dtype=float)
    lr_history = np.asarray(history["lr"], dtype=float)

    if len(val_loss) == 0:
        raise RuntimeError(f"{config_name}, seed={seed}: empty history.")

    best_idx = int(np.argmin(val_loss))
    best_round = best_idx + 1

    row = {
        "config_name": config_name,
        "seed": int(seed),
        "local_epochs": int(local_epochs),
        "clients_per_round": int(clients_per_round),
        "client_fraction": float(clients_per_round / len(client_loaders)),
        "early_stop_patience": int(early_stop_patience),
        "max_rounds": int(MAX_ROUNDS),
        "rounds_run": int(len(val_loss)),
        "best_round": int(best_round),
        "best_val_loss": float(val_loss[best_idx]),
        "best_lr": float(lr_history[best_idx]),
        "checkpoint_dir": str(checkpoint_dir),
    }

    for j, h in enumerate(HORIZONS):
        row[f"rmse_{h}"] = float(val_rmse[best_idx, j])
        row[f"mae_{h}"] = float(val_mae[best_idx, j])

    row["mean_rmse"] = float(np.mean(val_rmse[best_idx]))
    row["mean_mae"] = float(np.mean(val_mae[best_idx]))
    return row


def json_ready_history(history):
    clean = {}
    for key, values in history.items():
        if isinstance(values, np.ndarray):
            clean[key] = values.tolist()
        elif isinstance(values, list):
            clean[key] = [
                v.tolist() if isinstance(v, np.ndarray)
                else v.item() if isinstance(v, np.generic)
                else v
                for v in values
            ]
        elif isinstance(values, np.generic):
            clean[key] = values.item()
        else:
            clean[key] = values
    return clean


def save_state(results, histories):
    # Atomic-ish replacement: write temporary files first, then replace.
    tmp_csv = RUN_RESULTS_PATH.with_suffix(".tmp.csv")
    pd.DataFrame(results).to_csv(tmp_csv, index=False)
    tmp_csv.replace(RUN_RESULTS_PATH)

    tmp_json = HISTORIES_PATH.with_suffix(".tmp.json")
    with open(tmp_json, "w") as f:
        json.dump(histories, f, indent=2)
    tmp_json.replace(HISTORIES_PATH)


def run_key(config_name, seed):
    return (str(config_name), int(seed))


## 7. Resume existing Drive results if present

In [9]:

all_results = []
all_histories = {}

if RUN_RESULTS_PATH.exists():
    existing_df = pd.read_csv(RUN_RESULTS_PATH)
    all_results = existing_df.to_dict("records")
    print(f"Loaded {len(all_results)} completed runs from Drive.")
else:
    print("No existing run-level CSV found. Starting from zero.")

if HISTORIES_PATH.exists():
    with open(HISTORIES_PATH, "r") as f:
        all_histories = json.load(f)
    print("Loaded existing histories from Drive.")
else:
    print("No existing history JSON found.")

completed = {
    run_key(row["config_name"], row["seed"])
    for row in all_results
}

print("Completed unique runs:", len(completed))
print("Remaining runs:", N_RUNS - len(completed))

if len(completed) > N_RUNS:
    raise RuntimeError("Saved results contain more runs than the current tuning design.")


No existing run-level CSV found. Starting from zero.
No existing history JSON found.
Completed unique runs: 0
Remaining runs: 120


## 8. Run/resume all 120 tuning runs

In [10]:

run_counter = 0

for local_epochs in LOCAL_EPOCH_GRID:
    for clients_per_round in CLIENTS_PER_ROUND_GRID:
        for early_stop_patience in EARLY_STOP_PATIENCE_GRID:

            config_name = (
                f"E{local_epochs}_"
                f"C{clients_per_round}_"
                f"P{early_stop_patience}"
            )

            all_histories.setdefault(config_name, {})

            for seed in TUNING_SEEDS:
                run_counter += 1
                key = run_key(config_name, seed)

                if key in completed:
                    print(f"[{run_counter}/{N_RUNS}] SKIP completed: {config_name}, seed={seed}")
                    continue

                run_name = f"{config_name}_seed{seed}"
                checkpoint_dir = TUNING_DIR / "checkpoints" / config_name / f"seed_{seed}"
                checkpoint_dir.mkdir(parents=True, exist_ok=True)

                print("\n" + "=" * 88)
                print(f"[{run_counter}/{N_RUNS}] {run_name}")
                print(
                    f"E={local_epochs}, "
                    f"C={clients_per_round}/{len(client_loaders)}, "
                    f"P={early_stop_patience}, seed={seed}"
                )
                print("Checkpoint:", checkpoint_dir)
                print("=" * 88)

                set_seed(seed)

                model, history = fed.federated_train(
                    model_fn=model_fn,
                    client_loaders=client_loaders,
                    val_loader=val_loader,
                    cgm_mean=stats["cgm_mean"],
                    cgm_std=stats["cgm_std"],
                    device=DEVICE,

                    lr=LR,
                    weight_decay=WEIGHT_DECAY,
                    max_grad_norm=MAX_GRAD_NORM,

                    local_epochs=local_epochs,
                    clients_per_round=clients_per_round,
                    early_stop_patience=early_stop_patience,

                    max_rounds=MAX_ROUNDS,
                    lr_patience=LR_PATIENCE,
                    lr_factor=LR_FACTOR,

                    checkpoint_dir=checkpoint_dir,
                    verbose=True,
                    progress_bar=False,
                    progress_desc=run_name,
                    forward_fn=forward_fn,
                )

                row = summarise_run(
                    history=history,
                    seed=seed,
                    local_epochs=local_epochs,
                    clients_per_round=clients_per_round,
                    early_stop_patience=early_stop_patience,
                    config_name=config_name,
                    checkpoint_dir=checkpoint_dir,
                )

                all_results.append(row)
                all_histories[config_name][str(seed)] = json_ready_history(history)
                completed.add(key)

                # SAVE TO GOOGLE DRIVE AFTER EVERY SINGLE RUN.
                save_state(all_results, all_histories)

                print(
                    f"SAVED TO DRIVE: {run_name} | "
                    f"best_round={row['best_round']} | "
                    f"best_val_loss={row['best_val_loss']:.6f}"
                )
                print("Run-level CSV:", RUN_RESULTS_PATH)

print("\nTraining/resume loop complete.")
print("Completed unique runs:", len(completed))


流式输出内容被截断，只能显示最后 5000 行内容。
round  98/100  val_loss=0.4137  val_rmse(mg/dL)=[11.6, 18.7, 27.6, 33.1, 36.5]  lr=6.25e-06
round  99/100  val_loss=0.4137  val_rmse(mg/dL)=[11.6, 18.7, 27.6, 33.1, 36.5]  lr=6.25e-06
round 100/100  val_loss=0.4137  val_rmse(mg/dL)=[11.6, 18.7, 27.6, 33.1, 36.5]  lr=6.25e-06
SAVED TO DRIVE: E2_C30_P10_seed35 | best_round=100 | best_val_loss=0.413656
Run-level CSV: /content/drive/MyDrive/dissertation/experiments/7_federated_learning/results/configuration_tuning/fed_cgm_multiseed/run_level_results.csv

[66/120] E2_C30_P10_seed42
E=2, C=30/30, P=10, seed=42
Checkpoint: /content/drive/MyDrive/dissertation/experiments/7_federated_learning/results/configuration_tuning/fed_cgm_multiseed/checkpoints/E2_C30_P10/seed_42
device: cuda  model: GRUPredictor  params: 1,021,957  trainable: 1,021,957  clients: 30 (30/30 sampled per round)
round   1/100  val_loss=0.8642  val_rmse(mg/dL)=[38.5, 38.6, 39.2, 39.6, 39.9]  lr=1.00e-04
round   2/100  val_loss=0.7605  val_rmse(mg/dL)

## 9. Validate saved run-level results

In [11]:

# Reload from Drive rather than trusting in-memory variables.
assert RUN_RESULTS_PATH.exists(), f"Missing saved CSV: {RUN_RESULTS_PATH}"

run_results_df = pd.read_csv(RUN_RESULTS_PATH)

key_counts = (
    run_results_df
    .groupby(["config_name", "seed"])
    .size()
)

assert (key_counts == 1).all(), "Duplicate config/seed rows detected."

expected_configs = {
    f"E{e}_C{c}_P{p}"
    for e in LOCAL_EPOCH_GRID
    for c in CLIENTS_PER_ROUND_GRID
    for p in EARLY_STOP_PATIENCE_GRID
}

assert set(run_results_df["config_name"]) == expected_configs, (
    "Saved configurations do not match the intended grid."
)

counts_per_config = run_results_df.groupby("config_name")["seed"].nunique()

print("Saved run-level shape:", run_results_df.shape)
print("Unique configurations:", run_results_df["config_name"].nunique())
print("Unique config/seed runs:", len(key_counts))
print("\nSeeds per configuration:")
display(counts_per_config.to_frame("n_seeds"))

if len(run_results_df) != N_RUNS:
    raise RuntimeError(
        f"Expected {N_RUNS} completed runs, found {len(run_results_df)}. "
        "Re-run the previous cell to resume missing runs."
    )

assert (counts_per_config == len(TUNING_SEEDS)).all(), (
    "At least one configuration does not contain all 10 seeds."
)

print("\nAll 120 runs are present on Drive: PASS")


Saved run-level shape: (120, 24)
Unique configurations: 12
Unique config/seed runs: 120

Seeds per configuration:


,n_seeds
config_name,
E1_C15_P10,10
E1_C15_P20,10
E1_C30_P10,10
E1_C30_P20,10
E2_C15_P10,10
E2_C15_P20,10
E2_C30_P10,10
E2_C30_P20,10
E5_C15_P10,10



All 120 runs are present on Drive: PASS


## 10. Aggregate the 10 seeds within each configuration

In [12]:

group_cols = [
    "config_name",
    "local_epochs",
    "clients_per_round",
    "client_fraction",
    "early_stop_patience",
]

agg_spec = {
    "best_val_loss": ["mean", "std"],
    "best_round": ["mean", "std"],
    "rounds_run": ["mean", "std"],
    "mean_rmse": ["mean", "std"],
    "mean_mae": ["mean", "std"],
}

for h in HORIZONS:
    agg_spec[f"rmse_{h}"] = ["mean", "std"]
    agg_spec[f"mae_{h}"] = ["mean", "std"]

summary_df = (
    run_results_df
    .groupby(group_cols, as_index=False)
    .agg(agg_spec)
)

# Flatten MultiIndex columns.
summary_df.columns = [
    col[0] if isinstance(col, tuple) and col[1] == ""
    else f"{col[0]}_{col[1]}" if isinstance(col, tuple)
    else col
    for col in summary_df.columns
]

# Primary selection criterion:
# lowest mean of each seed's best validation loss.
summary_df = summary_df.sort_values(
    ["best_val_loss_mean", "mean_rmse_mean"],
    ascending=[True, True],
).reset_index(drop=True)

summary_df.insert(0, "rank", np.arange(1, len(summary_df) + 1))
summary_df.to_csv(SUMMARY_PATH, index=False)

print("Configuration-level summary saved to:")
print(SUMMARY_PATH)

display(summary_df)


Configuration-level summary saved to:
/content/drive/MyDrive/dissertation/experiments/7_federated_learning/results/configuration_tuning/fed_cgm_multiseed/configuration_summary_across_seeds.csv


,rank,config_name,local_epochs,clients_per_round,client_fraction,early_stop_patience,best_val_loss_mean,best_val_loss_std,best_round_mean,best_round_std,...,mae_60_mean,mae_60_std,rmse_90_mean,rmse_90_std,mae_90_mean,mae_90_std,rmse_120_mean,rmse_120_std,mae_120_mean,mae_120_std
0,1,E5_C15_P20,5,15,0.5,20,0.409169,0.009309,52.4,18.367846,...,18.348057,0.382460,32.877469,0.313039,22.054839,0.572881,36.325238,0.344542,24.570925,0.699512
1,2,E2_C15_P20,2,15,0.5,20,0.414536,0.006701,46.9,13.152524,...,18.664612,0.286302,32.841422,0.167872,22.190289,0.371167,35.900211,0.157907,24.528886,0.436293
2,3,E2_C30_P10,2,30,1.0,10,0.415809,0.002120,98.1,2.514403,...,18.226565,0.038174,33.152244,0.103584,21.699487,0.058903,36.519213,0.114044,24.004945,0.061152
3,4,E2_C30_P20,2,30,1.0,20,0.415809,0.002120,98.1,2.514403,...,18.226565,0.038174,33.152244,0.103584,21.699487,0.058903,36.519213,0.114044,24.004945,0.061152
4,5,E5_C30_P10,5,30,1.0,10,0.421660,0.007364,97.9,2.378141,...,18.595973,0.444703,33.406485,0.243080,22.083456,0.383538,36.545464,0.326682,24.287159,0.285574
5,6,E5_C30_P20,5,30,1.0,20,0.421660,0.007364,97.9,2.378141,...,18.595973,0.444703,33.406485,0.243080,22.083456,0.383538,36.545464,0.326682,24.287159,0.285574
6,7,E2_C15_P10,2,15,0.5,10,0.422165,0.006297,36.2,6.373556,...,18.885212,0.265188,33.039891,0.275171,22.394605,0.407443,36.065693,0.272007,24.731944,0.463614
7,8,E1_C30_P10,1,30,1.0,10,0.428953,0.002528,99.0,1.632993,...,18.647196,0.054571,33.493556,0.158869,22.049568,0.087167,36.722105,0.116583,24.314497,0.066741
8,9,E1_C30_P20,1,30,1.0,20,0.428953,0.002528,99.0,1.632993,...,18.647196,0.054571,33.493556,0.158869,22.049568,0.087167,36.722105,0.116583,24.314497,0.066741
9,10,E5_C15_P10,5,15,0.5,10,0.440261,0.013193,15.7,7.379401,...,19.431460,0.529910,33.772196,0.404466,23.323737,0.753148,37.109070,0.449597,26.050964,0.947142


## 11. Select and lock the FL configuration

In [13]:

best = summary_df.iloc[0]

selected_config = {
    "selection_metric": "lowest mean best validation loss across 10 tuning seeds",
    "tuning_seeds": TUNING_SEEDS,

    "local_epochs": int(best["local_epochs"]),
    "clients_per_round": int(best["clients_per_round"]),
    "client_fraction": float(best["client_fraction"]),
    "early_stop_patience": int(best["early_stop_patience"]),

    "max_rounds": MAX_ROUNDS,
    "lr_patience": LR_PATIENCE,
    "lr_factor": LR_FACTOR,

    "learning_rate": LR,
    "weight_decay": WEIGHT_DECAY,
    "batch_size": BATCH_SIZE,
    "max_grad_norm": MAX_GRAD_NORM,

    "gru_hidden_size": GRU_HIDDEN_SIZE,
    "gru_num_layers": GRU_NUM_LAYERS,
    "gru_dropout": GRU_DROPOUT,

    "mean_best_validation_loss": float(best["best_val_loss_mean"]),
    "sd_best_validation_loss": float(best["best_val_loss_std"]),
    "mean_best_round": float(best["best_round_mean"]),
    "sd_best_round": float(best["best_round_std"]),
}

with open(SELECTED_PATH, "w") as f:
    json.dump(selected_config, f, indent=2)

print("=" * 80)
print("SELECTED FED-CGM CONFIGURATION")
print("=" * 80)
print(f"Local epochs:          {selected_config['local_epochs']}")
print(
    f"Clients per round:     "
    f"{selected_config['clients_per_round']}/{len(client_loaders)}"
)
print(f"Early-stop patience:   {selected_config['early_stop_patience']}")
print(
    f"Mean best val loss:    "
    f"{selected_config['mean_best_validation_loss']:.6f} "
    f"± {selected_config['sd_best_validation_loss']:.6f}"
)
print(
    f"Mean best round:       "
    f"{selected_config['mean_best_round']:.1f} "
    f"± {selected_config['sd_best_round']:.1f}"
)

print("\nValidation RMSE across 10 seeds (mean ± SD, mg/dL):")
for h in HORIZONS:
    print(
        f"  {h:>3} min: "
        f"{best[f'rmse_{h}_mean']:.3f} "
        f"± {best[f'rmse_{h}_std']:.3f}"
    )

print("\nSaved selected configuration:")
print(SELECTED_PATH)

print("\nIMPORTANT:")
print("- Selection used validation data only.")
print("- Selection used the 10-seed mean, not the best individual run.")
print("- Mean best round is descriptive; formal runs still use the locked early-stop rule.")
print("- The test set has not been loaded or evaluated.")


SELECTED FED-CGM CONFIGURATION
Local epochs:          5
Clients per round:     15/30
Early-stop patience:   20
Mean best val loss:    0.409169 ± 0.009309
Mean best round:       52.4 ± 18.4

Validation RMSE across 10 seeds (mean ± SD, mg/dL):
   15 min: 11.624 ± 0.320
   30 min: 18.642 ± 0.369
   60 min: 27.329 ± 0.335
   90 min: 32.877 ± 0.313
  120 min: 36.325 ± 0.345

Saved selected configuration:
/content/drive/MyDrive/dissertation/experiments/7_federated_learning/results/configuration_tuning/fed_cgm_multiseed/selected_fl_configuration.json

IMPORTANT:
- Selection used validation data only.
- Selection used the 10-seed mean, not the best individual run.
- Mean best round is descriptive; formal runs still use the locked early-stop rule.
- The test set has not been loaded or evaluated.


## 12. Compact 12-configuration comparison

In [14]:

compact_cols = [
    "rank",
    "config_name",
    "local_epochs",
    "clients_per_round",
    "early_stop_patience",
    "best_val_loss_mean",
    "best_val_loss_std",
    "best_round_mean",
    "rounds_run_mean",
    "mean_rmse_mean",
    "mean_rmse_std",
]

display(summary_df[compact_cols])


,rank,config_name,local_epochs,clients_per_round,early_stop_patience,best_val_loss_mean,best_val_loss_std,best_round_mean,rounds_run_mean,mean_rmse_mean,mean_rmse_std
0,1,E5_C15_P20,5,15,20,0.409169,0.009309,52.4,72.1,25.359729,0.320606
1,2,E2_C15_P20,2,15,20,0.414536,0.006701,46.9,66.9,25.736608,0.293338
2,3,E2_C30_P10,2,30,10,0.415809,0.002120,98.1,100.0,25.584761,0.071856
3,4,E2_C30_P20,2,30,20,0.415809,0.002120,98.1,100.0,25.584761,0.071856
4,5,E5_C30_P10,5,30,10,0.421660,0.007364,97.9,100.0,25.785313,0.296695
5,6,E5_C30_P20,5,30,20,0.421660,0.007364,97.9,100.0,25.785313,0.296695
6,7,E2_C15_P10,2,15,10,0.422165,0.006297,36.2,46.2,26.035463,0.208510
7,8,E1_C30_P10,1,30,10,0.428953,0.002528,99.0,100.0,26.112902,0.091018
8,9,E1_C30_P20,1,30,20,0.428953,0.002528,99.0,100.0,26.112902,0.091018
9,10,E5_C15_P10,5,15,10,0.440261,0.013193,15.7,25.7,26.510721,0.501355


## 13. Final Drive verification

In [15]:

required_files = [
    RUN_RESULTS_PATH,
    HISTORIES_PATH,
    SUMMARY_PATH,
    SELECTED_PATH,
]

print("Final output verification:")
for path in required_files:
    print(f"{path.exists()}  {path}")

assert all(path.exists() for path in required_files), (
    "At least one required output file is missing."
)

print("\nAll required analysis files are safely present under PROJECT_ROOT.")
print("Main output directory:")
print(TUNING_DIR)


Final output verification:
True  /content/drive/MyDrive/dissertation/experiments/7_federated_learning/results/configuration_tuning/fed_cgm_multiseed/run_level_results.csv
True  /content/drive/MyDrive/dissertation/experiments/7_federated_learning/results/configuration_tuning/fed_cgm_multiseed/all_histories.json
True  /content/drive/MyDrive/dissertation/experiments/7_federated_learning/results/configuration_tuning/fed_cgm_multiseed/configuration_summary_across_seeds.csv
True  /content/drive/MyDrive/dissertation/experiments/7_federated_learning/results/configuration_tuning/fed_cgm_multiseed/selected_fl_configuration.json

All required analysis files are safely present under PROJECT_ROOT.
Main output directory:
/content/drive/MyDrive/dissertation/experiments/7_federated_learning/results/configuration_tuning/fed_cgm_multiseed
